In [11]:
import pandas as pd
import bm25s
import Stemmer  # optional: for stemming

# Create your corpus here
corpus = []
cc_df = pd.read_csv("../data/court_considerations.csv")
print("cc_df loaded.")
court_doc = [{'citation':citation, 'text':text} for citation,text in zip(cc_df['citation'], cc_df['text'])]
for d in court_doc:
    corpus.append(d['text'])


# optional: create a stemmer
stemmer = Stemmer.Stemmer("german")
# Tokenize the corpus and only keep the ids (faster and saves memory)
corpus_tokens = bm25s.tokenize(corpus, stopwords="de", stemmer=stemmer)
# Create the BM25 model and index the corpus
retriever = bm25s.BM25()
retriever.index(corpus_tokens)
# You can save the arrays to a directory...
retriever.save("../data/bm25/cc")

bm25_retriever = bm25s.BM25.load("../data/bm25/cc", load_corpus=False)
print("bm25_retriever loaded")

cc_df loaded.


Split strings:   0%|          | 0/1985178 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1985178 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/1985178 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1985178 [00:00<?, ?it/s]

bm25_retriever loaded


In [2]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [3]:
court_consideration_d = dict(zip(cc_df['citation'], cc_df['text']))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(cc_df['citation'], cc_df['text'])]

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")
valid_id_2_gold_citations_d = {}
valid_id_2_query_d = {}
for query_id, gold_citations, query in zip(valid_df['query_id'], valid_df['gold_citations'], valid_df['query2']):
    valid_id_2_gold_citations_d[query_id] = gold_citations.split(';')
    valid_id_2_query_d[query_id] = query
    
print("data loaded.")

data loaded.


In [4]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

print("index loaded")

DenseIndex.embeddings:  (2107648, 1024)
index loaded


In [5]:
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)
print("reranker loaded.")

reranker loaded.


In [17]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm
import bge_utils
import numpy as np

recall_citations_l_l = []
gold_citations_l_l = []
recall_hit_l_d = {}
gold_citation_l_d = {}
stemmer = Stemmer.Stemmer("german")

for query_id, query in valid_id_2_query_d.items():
    hits1 = dense_index.search_with_score(query, top_k=300)
    hits2 = sparse_index.search_with_score(query, top_k=300)
    _hits = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)

    query_tokens = bm25s.tokenize(query, stemmer=stemmer)
    results, scores = bm25_retriever.retrieve(query_tokens, k=300)

    _hit_bm25 = []
    for i in range(results.shape[1]):
        idx, score = results[0, i], scores[0, i]
        # print(f"Rank {i+1} (score: {score:.2f}): {doc}")
        text = court_doc[idx]['text']
        citation = court_doc[idx]['citation']
        _hit_bm25.append(({'citation':citation, "text":text}, score))

    _hits = hits_utils.merge_hits_with_score_l_by_max(_hits,_hit_bm25)

    all_hits = _hits.copy()
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    recall_hit_l_d[query_id] = all_hits

    gold_citations_l_l.append(valid_id_2_gold_citations_d[query_id])

    citation_id_set = set()
    for hit,score in _hits:
        citation_id_l = citation_utils.extract_citations_from_text(hit['text'])
        for citation_id in citation_id_l:
            citation_id_set.add(citation_id)
    recall_citations_l_l.append(list(citation_id_set))

r = metric_utils.cal_recall(recall_citations_l_l, gold_citations_l_l)
print(r)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[val_001] hits_merge.len: 768


...

[val_009] hits_merge.len: 830


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[val_010] hits_merge.len: 838
0.4331037823777524


In [23]:
from collections import defaultdict

def calc_citation_score_from_sorted_cc_l(cc_with_score_l, top_k=50):
    filter_citation_id_set = set()
    for hit,score in cc_with_score_l[:top_k]:
        citation_id_l = citation_utils.extract_citations_from_text(hit['text'])
        for citation_id in citation_id_l:
            filter_citation_id_set.add(citation_id)

    result = defaultdict(int)
    for hit,score in cc_with_score_l:
        citation_id_l = citation_utils.extract_citations_from_text(hit['text'])
        for citation_id in citation_id_l:
            if citation_id not in filter_citation_id_set:
                continue
            result[citation_id] += 1

    l = [(citation_id,score) for citation_id,score in result.items()]
    l.sort(key=lambda x:x[1], reverse=True)
    return [citation_id for citation_id,_ in l]

In [27]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

result_citation_l = []
gold_citations_l = []

for idx, (query_id, query) in tqdm(enumerate(valid_id_2_query_d.items()), total=len(valid_id_2_query_d)):
    recall_hits = recall_hit_l_d[query_id]
    cc_with_score_l = reranker_utils.rerank_by_batch_chunked2(reranker, query, [hit for hit,_ in recall_hits])
    sorted_citation_l = calc_citation_score_from_sorted_cc_l(cc_with_score_l, top_k=100)

    result_citation_l.append(sorted_citation_l)
    gold_citations_l.append(valid_id_2_gold_citations_d[query_id])

max_limit = None
max_r = None
max_p = None
max_f1 = 0
for limit in [5,10,15,20,25,30,35,40,45]:
    result_citation_l2 = []
    
    for r in result_citation_l:
        result_citation_l2.append(r[:limit])
        
    result = metric_utils.macro_f1(result_citation_l2, gold_citations_l[:len(result_citation_l2)])
    if max_f1 < result['macro_f1']:
        max_r = result['macro_recall']
        max_p = result['macro_precision']
        max_limit = limit
        max_f1 = result['macro_f1']
print(f"[{max_limit}] r:", max_r, ", p:", max_p, "f1:",max_f1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/10 [00:00<?, ?it/s]

[20] r: 0.09025898078529657 , p: 0.09 f1: 0.0853634670532703
